<a href="https://colab.research.google.com/github/Safwanach2002/ML/blob/main/KNN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# K-NEAREST NEIGHBORS (KNN) CLASSIFICATION

# Import Libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from google.colab import drive
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import confusion_matrix, accuracy_score, precision_score, recall_score, f1_score

# Mount Google Drive
drive.mount('/content/drive')


# 1. Load and Understand the Dataset

df = pd.read_excel('/content/drive/MyDrive/KNN_Dataset.xlsx')

print(df.head(10))

# Input Features and Target
X = df.drop('Category', axis=1)
y = df['Category']

print("Input Features:", X.columns)
print("Target Variable: Category")

# Class Distribution
print(y.value_counts())
sns.countplot(x=y)
plt.title("Category Distribution")
plt.show()

# Numerical Features
X = X.select_dtypes(include=np.number)
print("Numerical Features:", X.columns)


# 2. Distance Calculation and Nearest Neighbors

# Select a new smartphone record
new_phone = X.iloc[0].values

# Calculate Euclidean Distance
distances = np.sqrt(np.sum((X.values - new_phone)**2, axis=1))

result = pd.DataFrame({
    'Distance': distances,
    'Category': y
})

print("Distances:")
print(result.sort_values('Distance').head(10))

# Select K Neighbors
K = 3
nearest = result.sort_values('Distance').head(K)
print("Nearest Neighbors:")
print(nearest)

# Majority Voting
print("Predicted Category:", nearest['Category'].mode()[0])


# 3. Feature Scaling and KNN Working

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Calculate Scaled Distances
new_scaled = X_scaled[0]
scaled_distances = np.sqrt(np.sum((X_scaled - new_scaled)**2, axis=1))

scaled_result = pd.DataFrame({
    'Distance': scaled_distances,
    'Category': y
})

print("Nearest Neighbors After Scaling:")
print(scaled_result.sort_values('Distance').head(K))

# Input -> Scaling -> Distance -> Sorting -> K Neighbors
# -> Majority Voting -> Prediction


# 4. Implement KNN using Scikit-learn

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, random_state=42, stratify=y
)

# Standardize Features
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Train KNN Model
knn = KNeighborsClassifier(n_neighbors=3)
knn.fit(X_train, y_train)

# Prediction
y_pred = knn.predict(X_test)

# Confusion Matrix
cm = confusion_matrix(y_test, y_pred, labels=knn.classes_)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=knn.classes_, yticklabels=knn.classes_)
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix")
plt.show()

# Evaluation Metrics
print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred, average='weighted', zero_division=0))
print("Recall:", recall_score(y_test, y_pred, average='weighted', zero_division=0))
print("F1 Score:", f1_score(y_test, y_pred, average='weighted', zero_division=0))

# TP, TN, FP and FN are calculated for each class separately.


# 5. Selecting the Value of K

k_values = range(1, 11)
accuracies = []

for k in k_values:
    model = KNeighborsClassifier(n_neighbors=k)
    model.fit(X_train, y_train)
    accuracies.append(model.score(X_test, y_test))

plt.plot(k_values, accuracies, marker='o')
plt.xlabel("K Value")
plt.ylabel("Testing Accuracy")
plt.title("K Value vs Accuracy")
plt.show()

best_k = list(k_values)[np.argmax(accuracies)]
print("Best K:", best_k)

# Compare Small and Large K
small = KNeighborsClassifier(n_neighbors=1)
large = KNeighborsClassifier(n_neighbors=best_k)

small.fit(X_train, y_train)
large.fit(X_train, y_train)

print("Prediction with K=1:", small.predict(X_test[:5]))
print("Prediction with Best K:", large.predict(X_test[:5]))


# 6. New Prediction and Model Understanding

# Create Three New Smartphone Records
new_phones = X.iloc[:3].copy()

# Scale New Records
new_phones_scaled = scaler.transform(new_phones)

# Predict Category
predictions = large.predict(new_phones_scaled)

# Display Nearest Neighbors and Predictions
for i in range(3):
    distances = np.sqrt(np.sum((X_train - new_phones_scaled[i])**2, axis=1))
    nearest_indices = np.argsort(distances)[:best_k]

    print("\nNew Smartphone:", new_phones.iloc[i].values)
    print("Nearest Neighbor Categories:", y_train.iloc[nearest_indices].values)
    print("Predicted Category:", predictions[i])

# Small K is affected by nearby records.
# Large K considers more records and gives majority voting.
